# <font color="#418FDE" size="6.5" uppercase>**Decision Trees**</font>

>Last update: 20260710.
    
By the end of this Lecture, you will be able to:
- Explain how decision trees split engineering data into increasingly useful groups. 
- Implement a simple decision stump or shallow tree using manual split calculations. 
- Interpret tree rules and identify signs of overfitting in overly deep trees. 


## **1. Tree Split Intuition**

### **1.1. Choosing Better Splits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_06/Lecture_A/image_01_01.jpg?v=1783668301" width="250">



>* Good splits create more similar groups
>* Clearer branches reduce prediction confusion

>* Compare questions to find cleaner separations
>* Useful splits make later decisions easier

>* Choose context-driven splits capturing major variation
>* Refine branches without adding needless complexity



In [ ]:
#@title Python Code - Choosing Better Splits

# Compare candidate splits using simple impurity.
# Engineering labels represent pump failure outcomes.
# Lower weighted impurity means clearer branches.

# Import small numerical and plotting tools.
import numpy as np
import matplotlib.pyplot as plt

# Store vibration values in millimeters per second.
vibration = np.array([1.2, 1.8, 2.1, 2.7, 3.2, 3.8, 4.4, 5.1, 5.8, 6.4])

# Store temperature values in degrees Celsius.
temperature = np.array([42, 45, 47, 49, 52, 55, 57, 60, 63, 66])

# Store one for failed pumps.
failed = np.array([0, 0, 0, 0, 1, 0, 1, 1, 1, 1])

# Check that all arrays align.
assert vibration.size == temperature.size == failed.size

# Define Gini impurity for binary labels.
def gini(labels):
    labels = np.asarray(labels)
    if labels.size == 0:
        return 0.0

    p_fail = labels.mean()
    return 1 - p_fail**2 - (1 - p_fail)**2

# Score one threshold split.
def split_score(feature, threshold, labels):
    left = labels[feature <= threshold]
    right = labels[feature > threshold]

    n_total = labels.size
    left_weight = left.size / n_total
    right_weight = right.size / n_total

    return left_weight * gini(left) + right_weight * gini(right)

# List candidate engineering questions.
candidates = [
    ("vibration", vibration, 3.0),
    ("vibration", vibration, 4.0),
    ("temperature", temperature, 54.0),
    ("temperature", temperature, 61.0),
]

# Evaluate every candidate split.
results = []
for name, feature, threshold in candidates:
    score = split_score(feature, threshold, failed)
    results.append((score, name, threshold))

# Choose the split with lowest impurity.
best_score, best_name, best_threshold = min(results)

# Print compact teaching results.
print("Unsplit Gini impurity:", round(gini(failed), 3))
print("Best split:", best_name, "<=", best_threshold)
print("Weighted Gini after split:", round(best_score, 3))

# Show all candidate scores briefly.
for score, name, threshold in sorted(results):
    print(name, "<=", threshold, "score", round(score, 3))

# Select feature values for plotting.
plot_feature = vibration if best_name == "vibration" else temperature

# Create one visual split diagram.
plt.figure(figsize=(7, 4))
plt.scatter(plot_feature, failed, c=failed, cmap="coolwarm", s=90)
plt.axvline(best_threshold, color="black", linestyle="--")
plt.yticks([0, 1], ["no failure", "failure"])
plt.xlabel(best_name)
plt.ylabel("pump outcome")
plt.title("A better split makes branches less mixed")
plt.grid(True, alpha=0.3)
plt.show()



### **1.2. Left Branch Logic**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_06/Lecture_A/image_01_02.jpg?v=1783668299" width="250">



>* Left branches hold data meeting split conditions
>* Each split creates a more focused group

>* Left splits turn measurements into categories
>* Useful branches create clearer, predictable groups

>* Each left turn further narrows context
>* Tree paths act like diagnostic questions



In [ ]:
#@title Python Code - Left Branch Logic

# Decision trees ask simple engineering questions.
# Left branches collect rows satisfying conditions.
# We manually inspect one vibration split.

import pandas as pd
import matplotlib.pyplot as plt

# Create tiny pump vibration data.
data = pd.DataFrame({
    "pump": ["P1", "P2", "P3", "P4", "P5", "P6"],
    "vibration_mm_s": [1.2, 1.8, 2.1, 3.4, 4.0, 4.6],
    "status": ["healthy", "healthy", "healthy", "fail", "fail", "fail"]
})

# Choose one possible split threshold.
threshold = 2.5
left_mask = data["vibration_mm_s"] < threshold
right_mask = ~left_mask

# Count labels on each branch.
left_counts = data.loc[left_mask, "status"].value_counts()
right_counts = data.loc[right_mask, "status"].value_counts()

# Summarize the left branch logic.
print(f"Split question: vibration < {threshold} mm/s?")
print(f"Left branch pumps: {left_mask.sum()} of {len(data)}")
print(f"Left branch labels: {left_counts.to_dict()}")
print(f"Right branch labels: {right_counts.to_dict()}")

# Plot the split and branch membership.
colors = ["tab:blue" if value else "tab:orange" for value in left_mask]
plt.figure(figsize=(7, 3))
plt.scatter(data["vibration_mm_s"], data["pump"], c=colors, s=90)
plt.axvline(threshold, color="black", linestyle="--", label="split threshold")
plt.xlabel("Average vibration, mm/s")
plt.ylabel("Pump identifier")
plt.title("Left branch means vibration is below threshold")
plt.legend()
plt.tight_layout()
plt.show()



### **1.3. Right Branch Groups**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_06/Lecture_A/image_01_03.jpg?v=1783668303" width="250">



>* Right branches capture threshold-exceeding data groups
>* Grouped cases reveal useful engineering patterns

>* Right branches reveal shared engineering risks
>* Focused groups enable more targeted splits

>* Right branches depend on earlier split conditions
>* Check group size, stability, and plausibility



In [ ]:
#@title Python Code - Right Branch Groups

# Right branches collect threshold exceeding cases.
# Small engineering data clarifies tree intuition.
# Manual counts reveal useful grouped behavior.

import pandas as pd
import matplotlib.pyplot as plt

# Create tiny pump observations with engineering labels.
data = pd.DataFrame({
    "vibration_mm_s": [1.8, 2.1, 2.7, 3.2, 3.8, 4.4, 5.1, 5.8],
    "wear_risk": ["low", "low", "low", "low", "high", "high", "high", "high"]
})

# Choose one manual decision tree threshold.
threshold = 3.5
right_group = data[data["vibration_mm_s"] > threshold]
left_group = data[data["vibration_mm_s"] <= threshold]

# Count high risk cases in each branch.
right_high = (right_group["wear_risk"] == "high").sum()
left_high = (left_group["wear_risk"] == "high").sum()

# Compute simple branch purity percentages.
right_rate = 100 * right_high / len(right_group)
left_rate = 100 * left_high / len(left_group)

# Print concise interpretation of the split.
print(f"Split rule: vibration <= {threshold} mm/s goes left.")
print(f"Left branch: {len(left_group)} pumps, {left_rate:.0f}% high wear risk.")
print(f"Right branch: {len(right_group)} pumps, {right_rate:.0f}% high wear risk.")
print("Right branch groups warmer, riskier operating behavior here.")

# Plot the two branch groups visually.
colors = data["wear_risk"].map({"low": "steelblue", "high": "tomato"})
plt.figure(figsize=(7, 3))
plt.scatter(data["vibration_mm_s"], [1] * len(data), c=colors, s=90)

# Mark the threshold separating left and right.
plt.axvline(threshold, color="black", linestyle="--", label="split threshold")
plt.yticks([])
plt.xlabel("Pump vibration, millimeters per second")

# Add a compact title and legend.
plt.title("Right branch contains vibration above threshold")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()



## **2. Choosing Tree Splits**

### **2.1. Categorical Split Calculations**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_06/Lecture_A/image_02_01.jpg?v=1783668305" width="250">



>* Split data using category labels
>* Prefer categories that reveal outcome differences

>* Count outcomes within each category
>* Choose splits that create similar outcomes

>* Assign categories carefully to tree branches
>* Check sample sizes before trusting purity



In [ ]:
#@title Python Code - Categorical Split Calculations

# Manual categorical splits reveal useful engineering groups.
# This example uses tiny pump maintenance data.
# We compare impurity before and after splitting.

import pandas as pd
import matplotlib.pyplot as plt

# Create a small categorical engineering dataset.
data = pd.DataFrame({
    "mode": ["normal", "normal", "normal", "normal",
             "high load", "high load", "high load", "high load",
             "standby", "standby", "standby", "standby"],
    "failure": [0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0]
})

# Check the dataset is small and complete.
assert len(data) == 12 and data.isna().sum().sum() == 0

# Define Gini impurity for binary outcomes.
def gini(values):
    counts = values.value_counts()
    total = len(values)
    return 1 - sum((count / total) ** 2 for count in counts)

# Compute parent impurity before any split.
parent_gini = gini(data["failure"])

# Compute weighted child impurity by category.
summary = data.groupby("mode")["failure"].agg(["count", "sum"])
summary["failure_rate"] = summary["sum"] / summary["count"]
summary["gini"] = data.groupby("mode")["failure"].apply(gini)
summary["weight"] = summary["count"] / len(data)

# Calculate split quality using weighted impurity.
weighted_gini = (summary["weight"] * summary["gini"]).sum()
gini_gain = parent_gini - weighted_gini

# Print compact manual split results.
print(f"Parent Gini before split: {parent_gini:.3f}")
print(f"Weighted Gini after split: {weighted_gini:.3f}")
print(f"Gini improvement from mode split: {gini_gain:.3f}")
print("Failure rates by pump mode:")
print(summary[["count", "sum", "failure_rate"]].round(2))

# Plot category failure rates for interpretation.
ax = summary["failure_rate"].plot(kind="bar", color="steelblue")
ax.set_title("Failure rate by pump operating mode")
ax.set_xlabel("Pump operating mode")
ax.set_ylabel("Failure rate next week")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()



### **2.2. Candidate Split Calculations**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_06/Lecture_A/image_02_02.jpg?v=1783668307" width="250">



>* Test thresholds as possible tree rules
>* Choose splits that create clearer groups

>* Sort data and test possible thresholds
>* Choose splits that reduce uncertainty

>* Compare every threshold using the same steps
>* Manual splits build tree intuition and interpretability



In [ ]:
#@title Python Code - Candidate Split Calculations

# Manual split calculations reveal tree choices.
# Tiny engineering data keeps arithmetic visible.
# Gini impurity measures class mixing.

# Import small table and plotting tools.
import pandas as pd
import matplotlib.pyplot as plt

# Create vibration readings and failure labels.
data = pd.DataFrame({
    "vibration_mm_s": [1.2, 1.8, 2.1, 2.7, 3.4, 4.0, 4.6, 5.2],
    "failed": [0, 0, 0, 1, 0, 1, 1, 1],
})

# Confirm the tiny dataset shape.
assert data.shape == (8, 2)

# Define Gini impurity for binary labels.
def gini(labels):
    total = len(labels)
    if total == 0:
        return 0.0

    positives = sum(labels)
    p_one = positives / total
    p_zero = 1 - p_one
    return 1 - p_zero**2 - p_one**2

# Sort observations by the candidate feature.
sorted_data = data.sort_values("vibration_mm_s")
values = sorted_data["vibration_mm_s"].tolist()
labels = sorted_data["failed"].tolist()

# Build thresholds between neighboring readings.
thresholds = []
for left_value, right_value in zip(values[:-1], values[1:]):
    midpoint = (left_value + right_value) / 2
    thresholds.append(midpoint)

# Score each candidate threshold manually.
rows = []
for threshold in thresholds:
    left = sorted_data[sorted_data["vibration_mm_s"] < threshold]
    right = sorted_data[sorted_data["vibration_mm_s"] >= threshold]

    left_gini = gini(left["failed"].tolist())
    right_gini = gini(right["failed"].tolist())
    weighted = (len(left) * left_gini + len(right) * right_gini) / len(data)
    rows.append((threshold, len(left), len(right), weighted))

# Store results in a compact table.
results = pd.DataFrame(
    rows,
    columns=["threshold", "left_n", "right_n", "weighted_gini"],
)

# Select the lowest impurity split.
best_row = results.loc[results["weighted_gini"].idxmin()]
parent_gini = gini(labels)

# Print only the essential teaching results.
print("Parent Gini:", round(parent_gini, 3))
print(results.round(3).to_string(index=False))
print("Best rule: vibration_mm_s <", round(best_row["threshold"], 2))
print("Best weighted Gini:", round(best_row["weighted_gini"], 3))

# Plot candidate split quality once.
plt.figure(figsize=(7, 4))
plt.plot(results["threshold"], results["weighted_gini"], marker="o")
plt.axvline(best_row["threshold"], color="red", linestyle="--")
plt.xlabel("Candidate vibration threshold, mm/s")
plt.ylabel("Weighted Gini after split")
plt.title("Manual candidate split calculations")
plt.grid(True, alpha=0.3)
plt.show()



### **2.3. Ranking Candidate Splits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_06/Lecture_A/image_02_03.jpg?v=1783668309" width="250">



>* Rank splits by how well groups separate
>* Choose the most informative first question

>* Compare all splits using one fair score
>* Choose the split with clearest improvement

>* Balance split score with practical reliability
>* Prefer stable, interpretable features over noisy patterns



In [ ]:
#@title Python Code - Ranking Candidate Splits

# Rank candidate splits using manual impurity calculations.
# Small pump data keeps the example readable.
# Lower weighted Gini means a better split.

import pandas as pd
import matplotlib.pyplot as plt

# Create tiny engineering maintenance data.
data = pd.DataFrame({
    "temperature_c": [62, 68, 71, 75, 79, 83, 88, 91],
    "vibration_mm_s": [1.2, 1.5, 1.7, 2.1, 2.8, 3.2, 3.8, 4.1],
    "needs_maintenance": [0, 0, 0, 0, 1, 1, 1, 1],
})

# Confirm the dataset is intentionally small.
assert len(data) == 8 and data.shape[1] == 3

# Compute Gini impurity for one branch.
def gini(labels):
    counts = labels.value_counts()
    total = len(labels)

    if total == 0:
        return 0.0

    probabilities = counts / total
    return 1.0 - sum(probabilities ** 2)

# Score one candidate threshold split.
def weighted_gini(frame, feature, threshold):
    left = frame[frame[feature] <= threshold]
    right = frame[frame[feature] > threshold]

    left_weight = len(left) / len(frame)
    right_weight = len(right) / len(frame)

    left_score = gini(left["needs_maintenance"])
    right_score = gini(right["needs_maintenance"])

    return left_weight * left_score + right_weight * right_score

# List candidate engineering questions.
candidates = [
    ("temperature_c", 77),
    ("temperature_c", 85),
    ("vibration_mm_s", 2.5),
    ("vibration_mm_s", 3.5),
]

# Rank candidates by smallest weighted impurity.
rows = []
for feature, threshold in candidates:
    score = weighted_gini(data, feature, threshold)
    rows.append((feature, threshold, score))

# Store ranked split results.
ranked = pd.DataFrame(
    rows,
    columns=["feature", "threshold", "weighted_gini"],
).sort_values("weighted_gini")

# Print only the compact ranking table.
print("Candidate split ranking, lower Gini is better:")
print(ranked.to_string(index=False))

# Highlight the best first question.
best = ranked.iloc[0]
print(f"Best split: {best.feature} <= {best.threshold}")

# Plot the ranked impurity scores.
labels = ranked["feature"] + " <= " + ranked["threshold"].astype(str)
plt.figure(figsize=(7, 4))

plt.bar(labels, ranked["weighted_gini"], color="steelblue")
plt.ylabel("Weighted Gini impurity")
plt.title("Ranking candidate decision stump splits")

plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()



## **3. Tree Limits**

### **3.1. Overfitting Warning Signs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_06/Lecture_A/image_03_01.jpg?v=1783668315" width="250">



>* Deep narrow branches often signal memorization
>* Training accuracy may not generalize

>* High training accuracy can hide overfitting
>* Poor new-data performance signals memorized noise

>* Question overly specific, impractical tree rules
>* Prefer stable rules with meaningful leaf groups



In [ ]:
#@title Python Code - Overfitting Warning Signs

# Trees can memorize noisy engineering measurements.
# Validation checks reveal overfitting warning signs.
# Shallow rules often generalize more reliably.

import numpy as np
import matplotlib.pyplot as plt

# Create deterministic synthetic machine data.
rng = np.random.default_rng(7)
temperature = rng.normal(82, 9, 60)
vibration = rng.normal(4.8, 1.1, 60)

# Define failures using broad physical structure.
base_failure = (temperature > 86) | (vibration > 5.6)
noise = rng.random(60) < 0.12
failure = np.logical_xor(base_failure, noise).astype(int)

# Split data into training and validation sets.
train_idx = np.arange(0, 40)
valid_idx = np.arange(40, 60)

# Store features for simple manual trees.
X = np.column_stack([temperature, vibration])
y = failure

# Predict with one broad engineering rule.
def stump_predict(rows):
    return (X[rows, 0] > 86).astype(int)

# Predict with many narrow memorizing rules.
def deep_predict(rows):
    preds = stump_predict(rows)
    for i in train_idx:
        close_temp = np.abs(X[rows, 0] - X[i, 0]) < 1.2
        close_vib = np.abs(X[rows, 1] - X[i, 1]) < 0.18
        preds[close_temp & close_vib] = y[i]
    return preds

# Compute accuracy for both rule sets.
stump_train = np.mean(stump_predict(train_idx) == y[train_idx])
stump_valid = np.mean(stump_predict(valid_idx) == y[valid_idx])
deep_train = np.mean(deep_predict(train_idx) == y[train_idx])
deep_valid = np.mean(deep_predict(valid_idx) == y[valid_idx])

# Count tiny leaves as warning signs.
tiny_leaf_count = len(train_idx)
validation_gap = deep_train - deep_valid

# Print a compact overfitting comparison.
print(f"Shallow stump train accuracy: {stump_train:.2f}")
print(f"Shallow stump validation accuracy: {stump_valid:.2f}")
print(f"Deep memorizing tree train accuracy: {deep_train:.2f}")
print(f"Deep memorizing tree validation accuracy: {deep_valid:.2f}")
print(f"Deep tree validation gap: {validation_gap:.2f}")
print(f"Tiny training leaves created: {tiny_leaf_count}")
print("Warning: high train score plus lower validation score suggests overfitting.")

# Plot training data and the broad stump threshold.
plt.figure(figsize=(6, 4))
plt.scatter(temperature[train_idx], vibration[train_idx], c=y[train_idx], cmap="coolwarm", edgecolor="black")
plt.axvline(86, color="green", linestyle="--", label="simple split")
plt.xlabel("Operating temperature, degrees C")
plt.ylabel("Vibration, mm/s")
plt.title("Overfitting warning: many tiny rules memorize noise")
plt.legend()
plt.tight_layout()
plt.show()



### **3.2. Overfitting Signs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_06/Lecture_A/image_03_02.jpg?v=1783668311" width="250">



>* Overly deep trees create tiny data groups
>* Very specific rules may memorize noise

>* High training accuracy, poor validation performance
>* Deep rules may memorize sample noise

>* Deeper splits should make engineering sense
>* Fragile, narrow rules often signal noise



In [ ]:
#@title Python Code - Overfitting Signs

# Compare shallow and deep tree behavior.
# Tiny engineering data keeps output readable.
# Overfitting appears as fragile specific rules.

import numpy as np
import matplotlib.pyplot as plt

# Create deterministic sensor readings.
rng = np.random.default_rng(7)
temp_c = np.linspace(55, 95, 24)

# Build a realistic failure pattern.
vibration = rng.normal(3.0, 0.45, 24)
risk_score = temp_c + 8 * vibration

# Add two noisy inspection labels.
failed = risk_score > 105
failed[[5, 18]] = ~failed[[5, 18]]

# Split data into training and validation.
train_idx = np.arange(0, 16)
valid_idx = np.arange(16, 24)

# Define a simple stump predictor.
def stump_predict(values, threshold):
    return values > threshold

# Define a memorizing deep-tree style predictor.
def deep_predict(indices, train_indices, train_labels):
    lookup = dict(zip(train_indices.tolist(), train_labels.tolist()))
    return np.array([lookup.get(i, False) for i in indices])

# Choose one broad engineering split.
threshold = 78.0
train_stump = stump_predict(temp_c[train_idx], threshold)
valid_stump = stump_predict(temp_c[valid_idx], threshold)

# Simulate overly deep leaves memorizing training cases.
train_deep = deep_predict(train_idx, train_idx, failed[train_idx])
valid_deep = deep_predict(valid_idx, train_idx, failed[train_idx])

# Compute compact accuracy values.
train_stump_acc = np.mean(train_stump == failed[train_idx])
valid_stump_acc = np.mean(valid_stump == failed[valid_idx])
train_deep_acc = np.mean(train_deep == failed[train_idx])
valid_deep_acc = np.mean(valid_deep == failed[valid_idx])

# Print only the key teaching results.
print(f"Shallow rule: fail if temperature > {threshold:.0f} C")
print(f"Shallow train/validation accuracy: {train_stump_acc:.2f} / {valid_stump_acc:.2f}")
print("Deep rule: one tiny leaf per training machine")
print(f"Deep train/validation accuracy: {train_deep_acc:.2f} / {valid_deep_acc:.2f}")
print("Warning sign: perfect training fit but weak validation fit.")

# Plot labels and the broad split.
plt.figure(figsize=(7, 4))
plt.scatter(temp_c[~failed], vibration[~failed], label="passed", s=60)
plt.scatter(temp_c[failed], vibration[failed], label="failed", s=60)
plt.axvline(threshold, color="black", linestyle="--", label="shallow split")
plt.xlabel("Operating temperature, degrees C")
plt.ylabel("Vibration, millimeters per second")
plt.title("Overfitting sign: memorized leaves versus broad rule")
plt.legend()
plt.tight_layout()
plt.show()



### **3.3. Rule Reliability**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_06/Lecture_A/image_03_03.jpg?v=1783668313" width="250">



>* Reliable rules reflect real patterns
>* Trust grows with evidence and new cases

>* Check rules for breadth, stability, and meaning
>* Combine tree results with engineering judgment

>* Deep trees create fragile, overly specific rules
>* Trust rules supported by validation and simplicity



In [ ]:
#@title Python Code - Rule Reliability

# Rule reliability depends on evidence.
# Tiny leaves can mislead engineers.
# Validation checks reveal fragile rules.

import numpy as np
import pandas as pd

# Create a small deterministic inspection dataset.
rng = np.random.default_rng(7)
rows = 40
vibration = rng.normal(5.0, 1.2, rows)

# Add one unusual training coincidence.
temperature = rng.normal(72.0, 4.0, rows)
shift = rng.choice(["day", "night"], rows)
fail = vibration + temperature / 30 > 7.6

# Build a compact engineering table.
data = pd.DataFrame({"vibration": vibration, "temperature": temperature, "shift": shift, "fail": fail})
data.loc[[3, 17], "fail"] = True

# Split data into training and validation parts.
train = data.iloc[:28].copy()
valid = data.iloc[28:].copy()
assert len(train) > 10 and len(valid) > 5

# Define two readable tree rules manually.
broad_rule = train["vibration"] > 5.4
narrow_rule = (train["vibration"] > 6.1) & (train["shift"] == "night")

# Measure support and accuracy for each rule.
def rule_report(name, train_mask, valid_mask):
    train_support = int(train_mask.sum())
    valid_support = int(valid_mask.sum())
    train_acc = float((train.loc[train_mask, "fail"]).mean()) if train_support else 0.0
    valid_acc = float((valid.loc[valid_mask, "fail"]).mean()) if valid_support else 0.0
    return name, train_support, valid_support, train_acc, valid_acc

# Apply the same rules to validation data.
valid_broad = valid["vibration"] > 5.4
valid_narrow = (valid["vibration"] > 6.1) & (valid["shift"] == "night")
reports = [rule_report("Broad vibration rule", broad_rule, valid_broad), rule_report("Narrow deep-tree rule", narrow_rule, valid_narrow)]

# Print a compact reliability comparison.
print("Rule reliability check for inspection failures")
for name, train_n, valid_n, train_acc, valid_acc in reports:
    print(f"{name}: train n={train_n}, valid n={valid_n}, train fail rate={train_acc:.2f}, valid fail rate={valid_acc:.2f}")

# Summarize the interpretation lesson.
print("Reliable rules need support, validation performance, and engineering plausibility.")
print("Very narrow rules may fit coincidences instead of stable mechanisms.")



# <font color="#418FDE" size="6.5" uppercase>**Decision Trees**</font>


In this lecture, you learned to:
- Explain how decision trees split engineering data into increasingly useful groups. 
- Implement a simple decision stump or shallow tree using manual split calculations. 
- Interpret tree rules and identify signs of overfitting in overly deep trees. 

In the next Lecture (Lecture B), we will go over 'K-Means Clustering'